# 01 — Basic IP Geolocation Lookup with IP2Location.io

**Use case:** Look up a public IPv4 or IPv6 address and display its geographic and network information using the [IP2Location.io IP Geolocation API](https://www.ip2location.io/ip2location-documentation).

**You will learn to:**
1. Prepare a Python environment for API requests.
2. Configure an IP address and (optionally) an API key.
3. Send an HTTPS request to IP2Location.io.
4. Read selected fields and inspect the complete JSON response.

This notebook uses the public example IP `8.8.8.8`. You can change it to another public IPv4 or IPv6 address. **Location results are estimates, not precise device coordinates.**

## 1. Install the dependency

Run this cell once in your Jupyter environment. The `requests` package lets Python make HTTPS calls.

In [ ]:
%pip install requests

## 2. Configure the lookup

The API endpoint is `https://api.ip2location.io/`. An API key is **optional for a basic lookup**: the service currently permits a limited number of keyless requests.

For authenticated lookups, set the `IP2LOCATION_API_KEY` environment variable before starting Jupyter, or uncomment the optional `getpass()` lines below. The code sends your key in an **Authorization: Bearer** header rather than putting it into the URL.

Get an API key at [IP2Location.io](https://www.ip2location.io/) if needed. Avoid committing API keys or notebook outputs containing secrets to a public repository.

In [ ]:
import os
import ipaddress

API_URL = "https://api.ip2location.io/"
IP_ADDRESS = "8.8.8.8"  # Change this to the public IPv4 or IPv6 address to look up
API_KEY = os.environ.get("IP2LOCATION_API_KEY", "").strip()

# If you'd rather enter a key interactively without displaying it:
# from getpass import getpass
# API_KEY = getpass("IP2Location.io API key (optional): ").strip()

# Validate locally before consuming an API request.
ipaddress.ip_address(IP_ADDRESS)
print(f"Ready to look up {IP_ADDRESS} ({'authenticated' if API_KEY else 'keyless'} mode).")

## 3. Make the API request

The function below works for both IPv4 and IPv6. It sets a timeout, checks HTTP errors, and reports any error object returned by the API. The response fields depend on your subscription plan.

In [ ]:
import requests


def lookup_ip(ip_address: str, api_key: str = "") -> dict:
    """Return IP2Location.io geolocation data for a public IP address."""
    ipaddress.ip_address(ip_address)  # Raises ValueError if the IP is malformed.

    headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
    try:
        response = requests.get(
            API_URL,
            params={"ip": ip_address, "format": "json"},
            headers=headers,
            timeout=15,
        )
        response.raise_for_status()
    except requests.exceptions.RequestException as exc:
        raise RuntimeError(f"Could not complete the API request: {exc}") from exc

    try:
        payload = response.json()
    except ValueError as exc:
        raise RuntimeError("The API returned a non-JSON response.") from exc

    if not isinstance(payload, dict):
        raise RuntimeError("Unexpected API response format (expected a JSON object).")

    if "error" in payload:
        error = payload["error"]
        if isinstance(error, dict):
            code = error.get("error_code", "unknown")
            message = error.get("error_message", "Unknown API error")
            raise RuntimeError(f"IP2Location.io API error {code}: {message}")
        raise RuntimeError(f"IP2Location.io API error: {error}")

    return payload


result = lookup_ip(IP_ADDRESS, API_KEY)
print("Lookup completed successfully.")

## 4. Show the most useful fields

The free response normally includes location, coordinates, time zone, ASN and a basic proxy flag. Fields not supplied under the current plan are simply displayed as *Not available*.

In [ ]:
fields = {
    "ip": "IP address",
    "country_code": "Country code",
    "country_name": "Country",
    "region_name": "Region / state",
    "city_name": "City",
    "zip_code": "Postal code",
    "latitude": "Latitude",
    "longitude": "Longitude",
    "time_zone": "Time zone",
    "asn": "ASN",
    "as": "Autonomous system",
    "is_proxy": "Proxy flag",
}

for field, label in fields.items():
    value = result.get(field)
    print(f"{label:<20} : {value if value is not None else 'Not available'}")

## 5. Explore the full JSON response

The API may return additional fields depending on your plan. Inspect the raw response before deciding which fields to use in your own application.

In [ ]:
from IPython.display import JSON, display

display(JSON(data=result, expanded=True))

## 6. Try it yourself

- Change `IP_ADDRESS` in Section 2 to `1.1.1.1` and rerun Sections 2–5.
- Try a public IPv6 address, such as `2606:4700:4700::1111`.
- Compare the returned `country_name`, `asn`, and `is_proxy` values.

**API limits:** IP2Location.io currently documents up to **1,000 keyless queries per day** and a **Free Plan with up to 50,000 queries per month**. Consult the [official API documentation](https://www.ip2location.io/ip2location-documentation) for the latest limits, field availability, and error codes.

**Next use case suggestion:** Batch IP address lookups from a CSV file using the regular single-IP endpoint (without requiring the paid bulk endpoint).